# 07 - CTD reasoning with distractors and counterfactuals

Run this after Notebook 06 in the same Colab runtime. It reuses `model`, `tokenizer`, `eval_df`, and `generate_batch`.

This evaluates whether the SFT model follows explicit evidence rather than merely copying a familiar disease association. It adds distractor gene-disease edges and a counterfactual edge that deliberately changes the disease attached to the queried gene.

Important: counterfactual examples are synthetic diagnostic tests, not biomedical claims.

In [ ]:
import random
import re
import pandas as pd

required = ['model', 'tokenizer', 'eval_df', 'generate_batch']
missing = [x for x in required if x not in globals()]
if missing:
    raise RuntimeError(f'Run Notebook 06 first in the same runtime. Missing: {missing}')

def norm_text(s):
    return re.sub(r'[^a-z0-9]+', ' ', str(s).lower()).strip()


## 1. Build distractor questions

Each question contains the true Chemical -> Gene edge, the true Gene -> Disease edge, and several unrelated gene-disease edges. The model must identify the disease reachable through the queried gene.

In [ ]:
rng = random.Random(42)
pool = eval_df[['gene','disease']].drop_duplicates().to_dict('records')

distractor_rows = []
for _, row in eval_df.iterrows():
    candidates = [x for x in pool if x['gene'] != row['gene'] and x['disease'] != row['disease']]
    if len(candidates) < 3:
        continue
    distractors = rng.sample(candidates, 3)
    edges = [f"{row['gene']} -> {row['disease']}"] + [f"{x['gene']} -> {x['disease']}" for x in distractors]
    rng.shuffle(edges)
    prompt = (
        f"Evidence A: {row['chemical']} has a CTD chemical-gene relationship with {row['gene']}.\n"
        + 'Gene-disease evidence:\n- ' + '\n- '.join(edges)
        + f"\nQuestion: Using only the evidence above, which disease is connected to {row['chemical']} through gene {row['gene']}? "
          'Return `Disease: <name>` followed by `Path: Chemical -> Gene -> Disease`.'
    )
    distractor_rows.append({**row.to_dict(), 'test_prompt': prompt})

distractor_df = pd.DataFrame(distractor_rows).head(200)
print('Distractor examples:', len(distractor_df))
print(distractor_df.iloc[0]['test_prompt'])


In [ ]:
def score_distractors(df, predictions):
    disease_hits, path_hits = [], []
    for (_, row), pred in zip(df.iterrows(), predictions):
        p = norm_text(pred)
        disease_ok = norm_text(row['disease']) in p
        gene_ok = norm_text(row['gene']) in p
        chemical_ok = norm_text(row['chemical']) in p
        disease_hits.append(disease_ok)
        path_hits.append(disease_ok and gene_ok and chemical_ok)
    return {
        'distractor_disease_accuracy': sum(disease_hits) / len(disease_hits),
        'distractor_path_accuracy': sum(path_hits) / len(path_hits),
    }

distractor_predictions = generate_batch(model, distractor_df['test_prompt'].tolist(), max_new_tokens=72, batch_size=8)
distractor_metrics = score_distractors(distractor_df, distractor_predictions)
print(distractor_metrics)


## 2. Counterfactual evidence test

For each evaluation example, replace the true disease in the second hop with a different disease. A model that follows the supplied evidence should switch its answer to the synthetic counterfactual disease. This tests evidence sensitivity; it does not test biomedical factuality.

In [ ]:
all_diseases = eval_df['disease'].drop_duplicates().tolist()
cf_rows = []
rng = random.Random(123)
for _, row in eval_df.iterrows():
    alternatives = [d for d in all_diseases if d != row['disease']]
    if not alternatives:
        continue
    cf_disease = rng.choice(alternatives)
    prompt = (
        'This is a synthetic reasoning test. Follow the supplied evidence even if it conflicts with prior knowledge.\n'
        f"Evidence 1: {row['chemical']} has a relationship with gene {row['gene']}.\n"
        f"Evidence 2: In this hypothetical evidence set, gene {row['gene']} is linked to disease {cf_disease}.\n"
        f"Question: According only to this hypothetical evidence, what disease is connected to {row['chemical']} through gene {row['gene']}? "
        'Return `Disease: <name>` and a short path.'
    )
    item = row.to_dict()
    item['counterfactual_disease'] = cf_disease
    item['test_prompt'] = prompt
    cf_rows.append(item)

cf_df = pd.DataFrame(cf_rows).head(200)
print('Counterfactual examples:', len(cf_df))
print(cf_df.iloc[0]['test_prompt'])


In [ ]:
cf_predictions = generate_batch(model, cf_df['test_prompt'].tolist(), max_new_tokens=72, batch_size=8)

follow_cf = []
stick_original = []
for (_, row), pred in zip(cf_df.iterrows(), cf_predictions):
    p = norm_text(pred)
    follow_cf.append(norm_text(row['counterfactual_disease']) in p)
    stick_original.append(norm_text(row['disease']) in p)

counterfactual_metrics = {
    'counterfactual_follow_accuracy': sum(follow_cf) / len(follow_cf),
    'original_disease_leak_rate': sum(stick_original) / len(stick_original),
}
print(counterfactual_metrics)


## 3. Compare clean, distractor, and counterfactual behavior

`clean` comes from Notebook 06 if `after_metrics` is still available. Distractor accuracy measures robustness to irrelevant edges. Counterfactual-follow accuracy measures whether the model changes its answer when the supplied evidence changes.

In [ ]:
print('CTD REASONING DIAGNOSTICS')
print('-' * 72)
if 'after_metrics' in globals():
    print(f"Clean disease accuracy          : {after_metrics['disease_accuracy']:.3f}")
    print(f"Clean chain accuracy            : {after_metrics['chain_accuracy']:.3f}")
print(f"Distractor disease accuracy     : {distractor_metrics['distractor_disease_accuracy']:.3f}")
print(f"Distractor full-path accuracy   : {distractor_metrics['distractor_path_accuracy']:.3f}")
print(f"Counterfactual follow accuracy  : {counterfactual_metrics['counterfactual_follow_accuracy']:.3f}")
print(f"Original disease leak rate      : {counterfactual_metrics['original_disease_leak_rate']:.3f}")


In [ ]:
# Inspect a few failures.
for i in range(min(5, len(cf_df))):
    print('=' * 80)
    print('CHEMICAL:', cf_df.iloc[i]['chemical'])
    print('GENE:', cf_df.iloc[i]['gene'])
    print('TRUE CTD DISEASE:', cf_df.iloc[i]['disease'])
    print('SYNTHETIC COUNTERFACTUAL:', cf_df.iloc[i]['counterfactual_disease'])
    print('MODEL:', cf_predictions[i])


## Interpretation

A useful evidence-following model should retain high accuracy when distractor edges are added and should switch to the counterfactual disease when the explicit hypothetical evidence changes. High clean accuracy with low counterfactual-follow accuracy suggests memorization or reliance on prior associations rather than faithful evidence use. High original-disease leak rate is another warning sign.

This notebook intentionally keeps counterfactual statements clearly labeled as synthetic so they are not confused with CTD facts.